# Laboratorio 8 — Datos abiertos y APIs

María Paula Angulo Collantes

Danna Gabriela Castro

Dataset: *Servicio de conectividad a internet en instituciones educativas del departamento de Norte de Santander* (datos.gov.co)

## 1. Dataset seleccionado y registro de la fuente

| Campo | Registro |
|---|---|
| Tema | Educación |
| Portal | Datos Abiertos Colombia (datos.gov.co) |
| ID del dataset | `px6y-fznz` |
| Entidad publicadora | [ confirmar con el metadato del portal: ver celda siguiente ] |
| URL (API) | https://www.datos.gov.co/api/v3/views/px6y-fznz/query.json |
| Fecha de consulta | 15 de octubre de 2026 |

In [12]:

import requests
from datetime import datetime
import pandas as pd
import numpy as np

ID = "px6y-fznz"
URL = f"https://www.datos.gov.co/d/{ID}"

datos = requests.get(
    f"https://www.datos.gov.co/api/views/{ID}.json"
).json()

print("Fuente: Datos Abiertos de Colombia")
print("Entidad:", datos.get("attribution"))
print("Dataset:", datos.get("name"))
print("URL:", URL)
print("Fecha de consulta:", datetime.now())


Fuente: Datos Abiertos de Colombia
Entidad: Gobernación de Norte de Santander, Norte de Santander
Dataset: SERVICIO DE CONECTIVIDAD A INTERNET EN INSTITUCIONES EDUCATIVAS DEL DEPARTAMENTO NORTE DE SANTANDER
URL: https://www.datos.gov.co/d/px6y-fznz
Fecha de consulta: 2026-10-15 22:34:27.717894


## 2. Carga con `requests` y validación inicial de estructura

In [13]:
payload = {
    "query": "SELECT * LIMIT 1000",
    "page": {"pageSize": 1000, "pageNumber": 1},
}


try:
    r = requests.post(URL, json=payload, timeout=60)
    print("Status code:", r.status_code)
    r.raise_for_status()
    data = r.json()
    print("Tipo de la respuesta:", type(data))
    print("Llaves:", list(data.keys()) if isinstance(data, dict) else f"lista de {len(data)} registros")
except Exception as e:
    print("Falló la API:", e)
    data = None

Status code: 404
Falló la API: 404 Client Error: Not Found for url: https://www.datos.gov.co/d/px6y-fznz


In [14]:
if data is not None:
    registros = data.get("data", data) if isinstance(data, dict) else data
    df = pd.DataFrame(registros)
    origen = "API"
else:
    
    df = pd.read_csv("SERVICIO_DE_CONECTIVIDAD_A_INTERNET_EN_INSTITUCIONES_EDUCATIVAS_DEL_DEPARTAMENTO_NORTE_DE_SANTANDER_20261015.csv")
    origen = "CSV local"

print("Origen de los datos:", origen)
print("Dimensiones:", df.shape)
df.head()

Origen de los datos: CSV local
Dimensiones: (254, 13)


,ID,DEPARTAMENTO,MUNICIPIO,CODIGO DANE SEDE,NOMBRE SEDE EDUCATIVA,MATRÍCULA A BENEFICIAR POR SEDE EDUCATIVA,ZONA (URBANA/RURAL),Tecnología Última Milla\r\n(Cobre – Fibra – HFC – Radio – Satelital – Móvil – Inalámbrica),ANCHO DE BANDA (Mbps) \r\nCAPACIDAD PLAN DE DATOS (GB),CONETIVIDAD PARA ESTUDIANTES DE LA SEDE (PLANES MÓVILES - ZONA WIFI - CONECTIVIDAD EN SEDE),MESES DE SERVICIO,FECHA INICIO SERVICIO,FECHA FIN SERVICIO
0,1,NORTE DE SANTANDER,ABREGO,154003000262,SEDE VARONES,475.000,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022
1,2,NORTE DE SANTANDER,ABREGO,154003000921,SEDE INTEGRADA SAN RAFAEL,492.000,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022
2,3,NORTE DE SANTANDER,ABREGO,154003000840,SEDE SAN ANTONIO,270.000,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022
3,4,NORTE DE SANTANDER,ABREGO,154003001668,SEDE CARLOS JULIO TORRADO PEÑARANDA,1.600,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022
4,5,NORTE DE SANTANDER,ABREGO,154003000823,SEDE SANTA BARBARA,1.075,URBANA,Radio,20Mbps,CONECTIVIDAD EN SEDE,4,01 DE JULIO DE 2022,30 DE NOVIEMBRE DE 2022


In [15]:

df.info()
print()
print("Nulos por columna:")
print(df.isna().sum())
print()
print("Filas duplicadas:", df.duplicated().sum())
print()
print("Valores distintos por columna:")
print(df.nunique())

<class 'pandas.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 13 columns):
 #   Column                                                                                       Non-Null Count  Dtype  
---  ------                                                                                       --------------  -----  
 0   ID                                                                                           254 non-null    int64  
 1   DEPARTAMENTO                                                                                 254 non-null    str    
 2   MUNICIPIO                                                                                    254 non-null    str    
 3   CODIGO DANE SEDE                                                                             254 non-null    int64  
 4   NOMBRE SEDE EDUCATIVA                                                                        254 non-null    str    
 5   MATRÍCULA A BENEFICIAR POR SEDE EDUCATIVA      

In [16]:
def col(df, clave):
    return next(c for c in df.columns if clave.lower() in c.lower())

c_mun  = col(df, "municipio")
c_dane = col(df, "dane")
c_mat  = col(df, "matr")
c_zona = col(df, "zona")
c_tec  = col(df, "tecnolog")
c_bw   = col(df, "ancho")
c_meses = col(df, "meses")

df[c_mat] = pd.to_numeric(df[c_mat], errors="coerce")
df[c_meses] = pd.to_numeric(df[c_meses], errors="coerce")
df[[c_mun, c_zona, c_mat]].describe(include="all")

,MUNICIPIO,ZONA (URBANA/RURAL),MATRÍCULA A BENEFICIAR POR SEDE EDUCATIVA
count,254,254,254.000000
unique,38,2,NaN
top,OCAÑA,URBANA,NaN
freq,34,186,NaN
mean,NaN,NaN,270.097894
std,NaN,NaN,252.190544
min,NaN,NaN,1.075000
25%,NaN,NaN,54.500000
50%,NaN,NaN,192.000000
75%,NaN,NaN,404.000000


## 3. Unidad de análisis

Una fila = una sede educativa (no una institución completa, ni un estudiante, ni un contrato). Lo respalda el código DANE de sede: es único en las 254 filas. La unidad es la sede porque el servicio de conectividad se asigna sede por sede, y cada sede reporta su propia matrícula a beneficiar y su zona.

In [17]:
print("Filas:", len(df))
print("Códigos DANE de sede únicos:", df[c_dane].nunique(), "->", "es llave única" if df[c_dane].is_unique else "HAY REPETIDOS")
print("Municipios distintos:", df[c_mun].nunique())

Filas: 254
Códigos DANE de sede únicos: 254 -> es llave única
Municipios distintos: 38


## 4. Pregunta analítica y variables clave

¿Cómo se distribuye la matrícula beneficiada por el servicio de conectividad entre sedes urbanas y rurales, y en qué municipios se concentra?

Variables clave:
1. ZONA (URBANA/RURAL): segmenta las sedes.
2. MATRÍCULA A BENEFICIAR POR SEDE EDUCATIVA: tamaño de la población que se beneficia.
3. MUNICIPIO: ubicación geográfica de la sede.

Control de calidad previo: en el CSV exportado, 19 valores de matrícula aparecen con decimales (de 1.075 a 2.152). Un número de estudiantes no puede ser decimal; lo más probable es un separador de miles mal interpretado (1.075 sería 1075). Es una hipótesis, hay que confirmarla con el portal; si la API trae valores enteros, esta corrección no se activa.

In [18]:

no_enteros = df[df[c_mat] % 1 != 0]
print("Matrículas con decimales:", len(no_enteros))
df["matricula"] = np.where(df[c_mat] % 1 != 0, (df[c_mat] * 1000).round(), df[c_mat])
print("Rango corregido:", df["matricula"].min(), "-", df["matricula"].max())

Matrículas con decimales: 19
Rango corregido: 7.0 - 2152.0


In [20]:
resumen_zona = df.groupby(c_zona)["matricula"].agg(sedes="count", matricula_total="sum", promedio="mean", mediana="median").round(1)
resumen_zona["% de sedes"] = (resumen_zona["sedes"] / resumen_zona["sedes"].sum() * 100).round(1)
resumen_zona["% de matrícula"] = (resumen_zona["matricula_total"] / resumen_zona["matricula_total"].sum() * 100).round(1)
resumen_zona

,sedes,matricula_total,promedio,mediana,% de sedes,% de matrícula
ZONA (URBANA/RURAL),,,,,,
RURAL,68,5598.0,82.3,46.5,26.8,5.9
URBANA,186,89845.0,483.0,356.5,73.2,94.1


In [22]:
df.groupby(c_mun)["matricula"].agg(sedes="count", matricula_total="sum").sort_values("matricula_total", ascending=False).head(10)

,sedes,matricula_total
MUNICIPIO,,
OCAÑA,34,17312.0
VILLA DEL ROSARIO,24,16044.0
LOS PATIOS,18,10876.0
PAMPLONA,23,6973.0
TIBÚ,10,5040.0
ABREGO,12,4450.0
EL ZULIA,9,3650.0
EL TARRA,5,3277.0
CHINÁCOTA,6,2817.0


Lectura con los datos: hay 254 sedes y 38 municipios. Las sedes rurales son 68 (26.8 % de las sedes) pero concentran solo 5,9 % de la matrícula a beneficiar (5.598 de 95.443 estudiantes); la mediana rural es de 46,5 estudiantes por sede frente a 356,5 en las urbanas. Ocaña (17.312), Villa del Rosario (16.044) y Los Patios (10.876) encabezan la matrícula. Esto describe cómo se reparte el servicio, no si es suficiente.

## 5. Riesgos y limitaciones del dataset

1. Las variables de calidad no varían. En las 254 sedes la tecnología es Radio, el ancho de banda es 20Mbps, el tipo de servicio es CONECTIVIDAD EN SEDE, y el servicio dura 4 meses con las mismas fechas. Sin variabilidad no se puede comparar calidad entre sedes: el dataset sirve para ver cobertura y tamaño, no velocidad ni tecnología.
2. Es un contrato puntual, no el estado actual.El servicio va del 1 de julio al 30 de noviembre de 2022 y la consulta es de 2026. Además solo incluye las sedes beneficiadas por ese servicio, no necesariamente todas las sedes del departamento, así que no permite hablar de cobertura total.
3. Errores de formato en matrícula. 19 valores con decimales por un probable separador de miles; sin corregirlos, la matrícula total se subestima (68.605 frente a 95.443).
4. Es una cifra planeada, no de uso. "Matrícula a beneficiar" es la población objetivo; no dice cuántos estudiantes usaron el servicio ni si hubo interrupciones.
5. Zona no está definida en el dataset (¿con qué criterio se clasifica como rural?), lo que limita la comparación urbano-rural.